In [85]:
import numpy as np
from collections import Counter
import torch.nn
import pandas as pd
import matplotlib.pyplot as plt
import re

In [10]:
# load data
# same preprocessing as linear regression initial test

df = pd.read_csv('../resume_data.csv')

# remove bom character and whitespace from column names
df.columns = df.columns.str.strip().str.replace('﻿', '', regex=False)

# drop duplicate column
df = df.drop(columns=['responsibilities.1'])

# strip leading/trailing whitespace from all string values
df = df.map(lambda x: x.strip() if isinstance(x, str) else x)

# normalize all string values to lowercase
df = df.map(lambda x: x.lower() if isinstance(x, str) else x)

# replace empty strings and n/a with None
df = df.replace({'': None, 'n/a': None})

In [11]:
# select features and target

# columns to use as features
feature_cols = ['skills', 'job_position_name', 'educationaL_requirements', 'responsibilities']

# drop rows where any feature or target is missing
df_model = df[feature_cols + ['matched_score']].dropna()

# combine all text features into a single string per row
df_model['combined'] = df_model[feature_cols].apply(lambda row: ' '.join(row.values), axis=1)

print(f"rows after dropping missing: {df_model.shape[0]}")

rows after dropping missing: 9488


In [41]:
df_model['combined'].iloc[0]

"['big data', 'hadoop', 'hive', 'python', 'mapreduce', 'spark', 'java', 'machine learning', 'cloud', 'hdfs', 'yarn', 'core java', 'data science', 'c++', 'data structures', 'dbms', 'rdbms', 'informatica', 'talend', 'amazon redshift', 'microsoft azure'] senior software engineer b.sc in computer science & engineering from a reputed university. technical support\r\ntroubleshooting\r\ncollaboration\r\ndocumentation\r\nsystem monitoring\r\nsoftware deployment\r\ntraining & mentorship\r\nindustry trends\r\nfield visits"

In [169]:
# code taken from CS4120 Assignment 2
# relativise the word embeddings file to only keep relevant words (words that appear in the training set)
def relativize(file, outfile, word_counter):
    """
    Relativize the word vectors to the given dataset represented by word counts
    :param file: word vectors file
    :param outfile: output file
    :param word_counter: Counter of words occurring in train/dev/test data
    :return:
    """
    f = open(file, encoding="utf-8")
    o = open(outfile, 'w')
    voc = []
    for line in f:
        word = line[:line.find(' ')]
        if word_counter[word] > 0:
            # print("Keeping word vector for " + word)
            voc.append(word)
            o.write(line)
    for word in word_counter:
        if word not in voc:
            count = word_counter[word]
            if count > 1:
                print("Missing " + word + " with count " + repr(count))
    f.close()
    o.close()


def relativize_sentiment_data():
    # Count all words in the train, dev, and *test* sets. Note that this use of looking at the test set is legitimate
    # because we're not looking at the labels, just the words, and it's only used to cache computation that we
    # otherwise would have to do later anyway.
    word_counter = Counter()
    for ex in df_model['combined']:
        for word in re.split("\s|/|-|\(|,", ex): # split on spaces, -, /, (, and ','
            word = word.strip("[]'.)-•:\"!;â€‹") # remove all of these characters from strings
            if word == "": # if an empty string is created
                continue # pass on
            word_counter[word] += 1
    # Uncomment these to relativize vectors to the dataset
    relativize("embeddings/dolma_300_2024_1.2M.100_combined.txt", "embeddings/dolma_300_relativised.txt", word_counter)

In [170]:
relativize_sentiment_data()

Missing bachelor’s with count 678
Missing pytesseract with count 56
Missing bnbc with count 340
Missing pc's with count 28
Missing 10key with count 28
Missing sr.officer with count 339
Missing computersavvy with count 28
Missing tr069 with count 28
Missing 15+ with count 367
Missing kpi's with count 28
Missing dvp&r with count 28
Missing cbmet with count 28
Missing ct's with count 28
Missing amx4+ with count 28
Missing webscrapping with count 28
Missing nanovision with count 28
Missing cmpro with count 28
Missing p.m.a.s with count 28
Missing ezlabor with count 28
Missing innowi with count 28
Missing dukpt with count 28
Missing torquisesvn with count 28
Missing broadpos with count 28
Missing iccverify with count 28
Missing astrex with count 28
Missing certifynow with count 28
Missing certpro with count 28
Missing a+ with count 56
Missing chasepaymentech with count 28
Missing netservers with count 28
Missing sarbanesoxley with count 28
Missing tensorflowjs with count 56
Missing catsweb 

The above print statement shows the strings in the training set that do not appear in the embeddings. The embeddings that I downloaded are from https://nlp.stanford.edu/projects/glove/ and used the newest DOLMA embeddings with 220B tokens and 1.2 million words in the vocab. Once relativised, the file drops from 4.5GB to 9.5 MB in size. After splitting the strings and removing excess, I have found a number of words that don't appear in the embeddings. Some of these are useless (see "bacons's") and some would be useful to have ("bachelor's" or "businessintelligence"). This shows that pretrained embeddings probably aren't the best use for resume screening because of the rarety of some words in common language.